# Опыт 07: KV-кеш на checkpoint 50 000

**Меняем только вычисление инференса. Обучения здесь нет.** Прежний байтовый Transformer 2+2/128, веса 50 000, FP32, greedy, batch 32, cap 512. Закончившиеся строки пока остаются в batch.

1. В Colab выберите GPU **T4** и выполните ячейки по порядку. При отсутствии PyArrow выполните `%pip install pyarrow`.
2. Загрузите **`hw02-byte-20k-continue-10000-to-50000.zip`** из опыта 06. Из него читаются только checkpoint 50 000 и контрольные вопросы; train не скачивается.
3. Сначала проверяются логиты/маски/позиции, затем обычная и кешированная генерация на тех же 64 validation-вопросах (3 повторения).
4. Затем оба пути генерируют **все 1919 test-ответов**. Обычный путь может занять около 6 минут по прежней оценке; таймер продолжает работать и после 180 секунд. Это контрольный опыт, а не финальная сдача.
5. Верните ZIP результатов и отдельно этот IPYNB с outputs: **Файл → Скачать → Скачать IPYNB**.

Если проверка совпадения останавливает notebook, выполните последнюю ячейку скачивания: архив сохранит доступные диагностики. Генерируемые программы не исполняются; pass@1 не измеряется.

Кеш сохраняет K/V предыдущих позиций по слоям и постоянные K/V вопроса. В single-token decode будущих ключей нет; padding всё равно маскируется. [KV-кеш](https://huggingface.co/docs/transformers/main/cache_explanation), [SDPA и bool masks](https://docs.pytorch.org/docs/2.11/generated/torch.nn.functional.scaled_dot_product_attention.html).


In [ ]:
# 1. Fixed configuration and runtime. No optimizer or training loop.
import ast
import hashlib
import io
import json
import math
import statistics
import time
import zipfile
from pathlib import Path
from urllib.request import urlretrieve

import torch
from torch import nn
from torch.nn import functional as F
from torch.nn.utils.rnn import pad_sequence
import pyarrow.parquet as pq

PAD, BOS, EOS, VOCAB = 0, 1, 2, 259
D_MODEL, N_LAYERS, MAX_POSITIONS = 128, 2, 2048
GENERATION_CAP, INFERENCE_BATCH, VALIDATION_REPEATS = 512, 32, 3
EXPECTED_TEST_ROWS = 1919
LOGIT_ATOL, LOGIT_RTOL = 1e-4, 1e-4
torch.manual_seed(42)
assert torch.cuda.is_available(), 'Enable GPU / T4 in Colab'
device = torch.device('cuda')
gpu_name = torch.cuda.get_device_name(0)
assert 'T4' in gpu_name, 'This timing experiment requires a T4'

RESULTS_FILE = Path('hw02-kv-cache-results.json')
TEST_GENERATIONS_FILE = Path('hw02-kv-cache-test-generations.json')
ARCHIVE_FILE = Path('hw02-kv-cache-checkpoint-50000-results.zip')
SOURCE_ARCHIVE = Path('hw02-byte-20k-continue-10000-to-50000.zip')
SOURCE_ARCHIVE_SHA = 'a3c93ccc5995849518a31e10eeace2748343efbf5ccb6ac87f8c14b9f49f9195'
SOURCE_CHECKPOINT_SHA = '3dbd745c9662aa2a73b3832e6ed09990dd9f41974c3dd7172eda1f00d5007271'
SOURCE_JSON_SHA = '598f09e0f60d1f77a362bc6883b01ec4c8f3b032a7b0ed8380291c37195c7634'

def sha(raw):
    return hashlib.sha256(raw).hexdigest()
def data_hash(rows):
    return sha(json.dumps(rows, ensure_ascii=False, sort_keys=True).encode())
def sync():
    if device.type == 'cuda':
        torch.cuda.synchronize()
def save_results():
    temporary = RESULTS_FILE.with_suffix('.tmp')
    temporary.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
    temporary.replace(RESULTS_FILE)

result = {
    'experiment': '07-kv-cache-byte-20k-pairs-checkpoint-50000',
    'gpu': gpu_name, 'torch_version': torch.__version__, 'dtype': 'torch.float32',
    'config': {'checkpoint_step': 50000, 'tokenizer': 'UTF-8 bytes',
        'batch_size_inference': INFERENCE_BATCH, 'generation_cap': GENERATION_CAP,
        'validation_repeats': VALIDATION_REPEATS, 'decoding': 'greedy',
        'finished_rows_remain_in_batch': True, 'norm_first': False},
    'training_performed': False, 'functional_quality_evaluated': False,
    'run_finished': False, 'full_test_completed': False,
    'phase': 'setup', 'validation_runs': [], 'validation_examples': {},
    'validation_comparisons': [], 'test_runs': {},
    'timer_scope': 'question encoding, transfers, encoder/cache preparation, greedy, '
                   'CPU token conversion/decoding and AST/return/exact checks; '
                   'warmup, prefix checks, comparisons, printing, downloads and file writes excluded',
    'limit_scope': 'full test timed end-to-end through decoded/inspected answers; '
                   'official checker and final submission timer have not been specified',
    'runtime_flags': {'mha_fastpath_enabled': torch.backends.mha.get_fastpath_enabled(),
        'float32_matmul_precision': torch.get_float32_matmul_precision(),
        'flash_sdp_enabled': torch.backends.cuda.flash_sdp_enabled(),
        'mem_efficient_sdp_enabled': torch.backends.cuda.mem_efficient_sdp_enabled(),
        'math_sdp_enabled': torch.backends.cuda.math_sdp_enabled()},
}
save_results()
print('Device:', gpu_name, '| PyTorch:', torch.__version__, '| FP32')

In [ ]:
# 2. Upload the unchanged experiment-06 ZIP; inspect only known members.
if not SOURCE_ARCHIVE.exists():
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload one ZIP from experiment 06')
    SOURCE_ARCHIVE = Path(next(iter(uploaded)))
    uploaded.clear()
raw_archive = SOURCE_ARCHIVE.read_bytes()
assert sha(raw_archive) == SOURCE_ARCHIVE_SHA, 'Wrong source ZIP; use the completed experiment-06 archive'
with zipfile.ZipFile(io.BytesIO(raw_archive)) as z:
    assert z.testzip() is None, 'ZIP integrity check failed'
    raw_checkpoint = z.read('hw02-byte-20k-continue-step-50000.pt')
    raw_json = z.read('hw02-byte-20k-continue-results.json')
assert sha(raw_checkpoint) == SOURCE_CHECKPOINT_SHA
assert sha(raw_json) == SOURCE_JSON_SHA
previous_run = json.loads(raw_json)
checkpoint = torch.load(io.BytesIO(raw_checkpoint), map_location='cpu', weights_only=True)
assert checkpoint['steps'] == previous_run['actual_steps'] == 50000
assert previous_run['run_finished'] is True
assert checkpoint['config'] == previous_run['config']
for ck, dk in [('data_revision', 'revision'), ('train_sha256', 'train_sha256'),
               ('validation_sha256', 'validation_sha256')]:
    assert checkpoint[ck] == previous_run['data'][dk]
assert {int(s['step'].item()) for s in checkpoint['optimizer']['state'].values()} == {50000}
assert all(bool(torch.isfinite(t).all()) for t in checkpoint['model'].values())
validation = [{'question': e['question'], 'code': e['reference_code']}
              for e in previous_run['diagnostics']['50000']['validation_64']['examples']]
assert len(validation) == 64 and data_hash(validation) == checkpoint['validation_sha256']
result['source'] = {'archive_filename': SOURCE_ARCHIVE.name, 'archive_sha256': SOURCE_ARCHIVE_SHA,
    'checkpoint_sha256': SOURCE_CHECKPOINT_SHA, 'source_results_sha256': SOURCE_JSON_SHA,
    'checkpoint_config': checkpoint['config'], 'data_revision': checkpoint['data_revision'],
    'train_sha256': checkpoint['train_sha256'], 'validation_sha256': checkpoint['validation_sha256'],
    'source_gpu': previous_run['gpu'], 'source_torch_version': previous_run['torch_version']}
result['previous_validation64'] = {
    'origin': 'saved user experiment-06 results, not a new measurement',
    'summary': previous_run['diagnostics']['50000']['validation_64']['summary']}
result['phase'] = 'checkpoint_verified'
save_results()
del raw_archive, raw_checkpoint, raw_json
print('Verified checkpoint 50000 and the same validation64; no training data downloaded.')

In [ ]:
# 3. The pinned public test, in its original row order; no labels or training.
revision = checkpoint['data_revision']
test_file = Path('hw02-data/test.parquet')
test_file.parent.mkdir(exist_ok=True)
test_url = f'https://huggingface.co/datasets/SlavaYus/tinysms_qa_12m/resolve/{revision}/data/test/test.parquet'
if not test_file.exists():
    urlretrieve(test_url, test_file)
test_sha = sha(test_file.read_bytes())
assert test_sha == '8bde59c8d8bb7edc4e8a503be22fc132c4a47773025d333432e78c5489e6522a'
test_questions = pq.read_table(test_file)['question'].to_pylist()
assert len(test_questions) == EXPECTED_TEST_ROWS
assert all(len(q.encode('utf-8')) + 2 <= MAX_POSITIONS for q in test_questions)
test_rows = [{'question': q} for q in test_questions]
result['test_data'] = {'url': test_url, 'file_sha256': test_sha, 'rows': len(test_rows),
    'questions_sha256': data_hash(test_rows), 'row_order': 'unchanged parquet order'}
save_results()
print('Validation:', len(validation), '| Test:', len(test_rows))

In [ ]:
# 4. Exact reference class from experiment 06; the same weights serve both paths.
def encode(text):
    return [b + 3 for b in text.encode('utf-8')]

def decode(ids):
    return bytes(i - 3 for i in ids if 3 <= i < VOCAB).decode('utf-8', errors='replace')

def source_batch(questions):
    return pad_sequence([torch.tensor([BOS] + encode(q) + [EOS]) for q in questions],
                        batch_first=True, padding_value=PAD).to(device)

class SimpleTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.tokens = nn.Embedding(VOCAB, D_MODEL, padding_idx=PAD)
        self.positions = nn.Embedding(MAX_POSITIONS, D_MODEL)
        self.transformer = nn.Transformer(d_model=D_MODEL, nhead=4,
            num_encoder_layers=N_LAYERS, num_decoder_layers=N_LAYERS,
            dim_feedforward=4 * D_MODEL, dropout=0.1, batch_first=True)
        self.transformer.encoder.enable_nested_tensor = False
        self.transformer.encoder.use_nested_tensor = False
        self.head = nn.Linear(D_MODEL, VOCAB)
    def embed(self, ids):
        if ids.shape[1] > MAX_POSITIONS:
            raise ValueError('Sequence too long; no silent truncation')
        pos = torch.arange(ids.shape[1], device=ids.device)
        return self.tokens(ids) * math.sqrt(D_MODEL) + self.positions(pos)
    def decode_prefix(self, tgt, memory, source_padding):
        mask = torch.ones(tgt.shape[1], tgt.shape[1], device=tgt.device, dtype=torch.bool).triu(1)
        return self.transformer.decoder(self.embed(tgt), memory, tgt_mask=mask,
            tgt_key_padding_mask=tgt.eq(PAD), memory_key_padding_mask=source_padding)
    def forward(self, src, tgt):
        memory = self.transformer.encoder(self.embed(src), src_key_padding_mask=src.eq(PAD))
        return self.head(self.decode_prefix(tgt, memory, src.eq(PAD)))
    @torch.inference_mode()
    def generate(self, src, max_new_tokens=512):
        self.eval()
        memory = self.transformer.encoder(self.embed(src), src_key_padding_mask=src.eq(PAD))
        tgt = torch.full((len(src), 1), BOS, device=src.device, dtype=torch.long)
        done = torch.zeros(len(src), device=src.device, dtype=torch.bool)
        for _ in range(max_new_tokens):
            hidden = self.decode_prefix(tgt, memory, src.eq(PAD))
            logits = self.head(hidden[:, -1])
            logits[:, [PAD, BOS]] = -torch.inf
            token = logits.argmax(-1).masked_fill(done, PAD)
            done |= token.eq(EOS)
            tgt = torch.cat([tgt, token[:, None]], dim=1)
            if bool(done.all()):
                break
        return tgt[:, 1:]

model = SimpleTransformer().to(device=device, dtype=torch.float32).eval()
model.load_state_dict(checkpoint['model'], strict=True)
assert sum(p.numel() for p in model.parameters()) == 1254915
assert all(p.dtype == torch.float32 for p in model.parameters())

def model_fingerprint():
    h = hashlib.sha256()
    for name, tensor in sorted(model.state_dict().items()):
        h.update(name.encode())
        h.update(tensor.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()

initial_model_fingerprint = model_fingerprint()
result['parameters'] = 1254915
result['model_fingerprint_before'] = initial_model_fingerprint
del checkpoint
save_results()
print('Parameters:', result['parameters'], '| optimizer is not created')

In [ ]:
# Incremental path for the existing Post-LN decoder; no new model parameters.
class CachedDecoder:
    @torch.inference_mode()
    def __init__(self, model, src, capacity):
        model.eval()
        if not 0 < capacity <= model.positions.num_embeddings:
            raise ValueError('Invalid cache capacity')
        self.model, self.capacity, self.position = model, capacity, 0
        self.batch_size = len(src)
        self.width = model.tokens.embedding_dim
        self.heads = model.transformer.decoder.layers[0].self_attn.num_heads
        self.head_dim = self.width // self.heads
        memory = model.transformer.encoder(model.embed(src), src_key_padding_mask=src.eq(PAD))
        # SDPA bool masks use True for allowed keys (opposite to MHA padding masks).
        self.source_allowed = src.ne(PAD)[:, None, None, :]
        self.valid = torch.zeros(len(src), capacity, dtype=torch.bool, device=src.device)
        self.position_vectors = model.positions(torch.arange(capacity, device=src.device))
        self.layers = []
        for layer in model.transformer.decoder.layers:
            if layer.norm_first:
                raise ValueError('This experiment implements the saved Post-LN model')
            for attention in (layer.self_attn, layer.multihead_attn):
                if (not attention._qkv_same_embed_dim or attention.num_heads != self.heads
                        or attention.bias_k is not None or attention.bias_v is not None
                        or attention.add_zero_attn):
                    raise ValueError('Unsupported attention configuration')
            attention = layer.multihead_attn
            projected_kv = F.linear(memory, attention.in_proj_weight[self.width:],
                                    attention.in_proj_bias[self.width:])
            cross_k, cross_v = projected_kv.chunk(2, dim=-1)
            shape = (len(src), self.heads, capacity, self.head_dim)
            self.layers.append({
                'self_k': memory.new_empty(shape), 'self_v': memory.new_empty(shape),
                'cross_k': self.split_heads(cross_k), 'cross_v': self.split_heads(cross_v),
            })

    def split_heads(self, values):
        return values.reshape(self.batch_size, -1, self.heads, self.head_dim).transpose(1, 2)

    def attend(self, attention, q, k, v, allowed):
        # One query at the latest position; all supplied keys are from its past/current.
        # is_causal=True would use an upper-left 1 x T mask and incorrectly hide the past.
        output = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed,
                                                dropout_p=0.0, is_causal=False)
        output = output.transpose(1, 2).contiguous().reshape(self.batch_size, 1, self.width)
        return attention.out_proj(output)

    @torch.inference_mode()
    def step(self, token_ids):
        if self.position >= self.capacity:
            raise ValueError('Cache capacity exceeded')
        if token_ids.ndim != 1 or len(token_ids) != self.batch_size:
            raise ValueError('One token per batch row is required')
        t = self.position
        x = self.model.tokens(token_ids[:, None]) * math.sqrt(self.width)
        x = x + self.position_vectors[t][None, None, :]
        self.valid[:, t] = token_ids.ne(PAD)
        self_allowed = self.valid[:, None, None, :t + 1]
        for layer, cache in zip(self.model.transformer.decoder.layers, self.layers):
            attention = layer.self_attn
            q, k, v = F.linear(x, attention.in_proj_weight, attention.in_proj_bias).chunk(3, dim=-1)
            cache['self_k'][:, :, t:t + 1] = self.split_heads(k)
            cache['self_v'][:, :, t:t + 1] = self.split_heads(v)
            self_output = self.attend(attention, self.split_heads(q),
                cache['self_k'][:, :, :t + 1], cache['self_v'][:, :, :t + 1], self_allowed)
            x = layer.norm1(x + layer.dropout1(self_output))

            attention = layer.multihead_attn
            q = F.linear(x, attention.in_proj_weight[:self.width], attention.in_proj_bias[:self.width])
            cross_output = self.attend(attention, self.split_heads(q),
                                      cache['cross_k'], cache['cross_v'], self.source_allowed)
            x = layer.norm2(x + layer.dropout2(cross_output))
            feed_forward = layer.linear2(layer.dropout(layer.activation(layer.linear1(x))))
            x = layer.norm3(x + layer.dropout3(feed_forward))

        if self.model.transformer.decoder.norm is not None:
            x = self.model.transformer.decoder.norm(x)
        self.position += 1
        return self.model.head(x[:, 0])


@torch.inference_mode()
def generate_cached(model, src, max_new_tokens=512):
    model.eval()
    cache = CachedDecoder(model, src, max_new_tokens)
    # The same stopping, PAD/BOS bans, torch.cat and finished rows as the old path.
    tgt = torch.full((len(src), 1), BOS, device=src.device, dtype=torch.long)
    done = torch.zeros(len(src), device=src.device, dtype=torch.bool)
    for _ in range(max_new_tokens):
        logits = cache.step(tgt[:, -1])
        logits[:, [PAD, BOS]] = -torch.inf
        token = logits.argmax(-1).masked_fill(done, PAD)
        done |= token.eq(EOS)
        tgt = torch.cat([tgt, token[:, None]], dim=1)
        if bool(done.all()):
            break
    return tgt[:, 1:]

In [ ]:
# 6. Incremental vs full-prefix logits: positions and unequal-length PAD masks.
@torch.inference_mode()
def check_prefix_logits(rows):
    src = source_batch([r['question'] for r in rows])
    tgt = pad_sequence([torch.tensor([BOS] + encode(r['code']) + [EOS]) for r in rows],
                       batch_first=True, padding_value=PAD).to(device)
    memory = model.transformer.encoder(model.embed(src), src_key_padding_mask=src.eq(PAD))
    cache = CachedDecoder(model, src, tgt.shape[1])
    positions = {0, 1, 2, 7, 31, 63, 127, 191, tgt.shape[1] - 1}
    checks = []
    for t in range(tgt.shape[1]):
        actual = cache.step(tgt[:, t])
        if t in positions:
            expected = model.head(model.decode_prefix(tgt[:, :t + 1], memory, src.eq(PAD))[:, -1])
            checks.append({'position': t, 'max_abs_error': (actual - expected).abs().max().item(),
                'all_close': bool(torch.isclose(actual, expected, atol=LOGIT_ATOL, rtol=LOGIT_RTOL).all())})
    return {'examples': len(rows), 'atol': LOGIT_ATOL, 'rtol': LOGIT_RTOL,
            'positions': checks, 'all_close': all(c['all_close'] for c in checks)}

result['phase'] = 'prefix_check'
save_results()
result['prefix_check'] = check_prefix_logits(validation[:3])
result['weights_unchanged_after_prefix_check'] = model_fingerprint() == initial_model_fingerprint
save_results()
print('PREFIX_CHECK', json.dumps(result['prefix_check']))
if not result['prefix_check']['all_close'] or not result['weights_unchanged_after_prefix_check']:
    result['phase'] = 'prefix_check_failed'
    save_results()
    raise RuntimeError('Prefix check failed; run the download cell and return the ZIP')

In [ ]:
# 7. Common measurement and diagnostics; generated programs are never executed.
def solve_value_return(tree):
    def contains(node):
        if isinstance(node, ast.Return):
            return node.value is not None and not (isinstance(node.value, ast.Constant) and node.value.value is None)
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef, ast.Lambda)):
            return False
        return any(contains(child) for child in ast.iter_child_nodes(node))
    return any(isinstance(n, ast.FunctionDef) and n.name == 'solve'
               and any(contains(stmt) for stmt in n.body) for n in tree.body)

def inspect_ids(row, ids):
    active = [int(i) for i in ids if i != PAD]
    code = decode(active)
    try:
        tree = ast.parse(code)
        parses = True
        defines_solve = any(isinstance(n, ast.FunctionDef) and n.name == 'solve' for n in tree.body)
        value_return = solve_value_return(tree)
    except (SyntaxError, ValueError):
        parses, defines_solve, value_return = False, False, False
    e = {'question': row['question'], 'token_ids': active, 'generated_code': code,
        'generated_tokens': len(active), 'eos_emitted': EOS in active,
        'reached_cap_without_eos': EOS not in active and len(active) == GENERATION_CAP,
        'parses_as_python': parses, 'defines_solve': defines_solve,
        'solve_has_value_return': value_return}
    if 'code' in row:
        e.update({'reference_code': row['code'], 'exact_code_match': code == row['code'],
                  'exact_tokens_with_eos': active == encode(row['code']) + [EOS]})
    return e

@torch.inference_mode()
def run_benchmark(path, rows, split, repeat):
    sync()
    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    examples = []
    for offset in range(0, len(rows), INFERENCE_BATCH):
        batch = rows[offset:offset + INFERENCE_BATCH]
        src = source_batch([r['question'] for r in batch])
        if path == 'reference':
            ids = model.generate(src, max_new_tokens=GENERATION_CAP)
        elif path == 'cached':
            ids = generate_cached(model, src, max_new_tokens=GENERATION_CAP)
        else:
            raise ValueError('Unknown generation path')
        examples.extend(inspect_ids(row, tokens) for row, tokens in zip(batch, ids.cpu().tolist()))
    sync()
    seconds = time.perf_counter() - started
    lengths = [e['generated_tokens'] for e in examples]
    summary = {'path': path, 'split': split, 'repeat': repeat, 'examples': len(rows),
        'seconds': seconds, 'answers_per_second': len(rows) / seconds,
        'generated_tokens': sum(lengths), 'tokens_per_second': sum(lengths) / seconds,
        'mean_generated_tokens': sum(lengths) / len(lengths), 'max_generated_tokens': max(lengths),
        'peak_gpu_memory_mib': torch.cuda.max_memory_allocated() / 2**20 if device.type == 'cuda' else None}
    for metric in ['eos_emitted', 'reached_cap_without_eos', 'parses_as_python', 'defines_solve',
                   'solve_has_value_return', 'exact_code_match', 'exact_tokens_with_eos']:
        if metric in examples[0]:
            summary[metric + '_count'] = sum(e[metric] for e in examples)
    return {'summary': summary, 'examples': examples}

def compare_examples(reference, cached):
    assert len(reference) == len(cached)
    mismatches = []
    for i, (a, b) in enumerate(zip(reference, cached)):
        assert a['question'] == b['question']
        if a['token_ids'] != b['token_ids']:
            first = next((j for j, (x, y) in enumerate(zip(a['token_ids'], b['token_ids'])) if x != y),
                         min(len(a['token_ids']), len(b['token_ids'])))
            mismatches.append({'index': i, 'first_different_token_index': first,
                'reference_length': len(a['token_ids']), 'cached_length': len(b['token_ids'])})
    return {'examples': len(reference), 'identical_token_sequences': len(reference) - len(mismatches),
        'identical_code_texts': sum(a['generated_code'] == b['generated_code'] for a, b in zip(reference, cached)),
        'all_tokens_equal': not mismatches, 'mismatches': mismatches}

def examples_hash(examples):
    return sha(json.dumps([e['token_ids'] for e in examples]).encode())

In [ ]:
# 8. Full-length warmup, then three alternating-order validation64 trials.
assert result['prefix_check']['all_close']
result['phase'] = 'validation_warmup'
save_results()
with torch.inference_mode():
    warm_src = source_batch([r['question'] for r in validation[:INFERENCE_BATCH]])
    model.generate(warm_src, max_new_tokens=GENERATION_CAP)
    generate_cached(model, warm_src, max_new_tokens=GENERATION_CAP)
    del warm_src
sync()

for repeat in range(VALIDATION_REPEATS):
    order = ('reference', 'cached') if repeat % 2 == 0 else ('cached', 'reference')
    trial = {}
    for path in order:
        result['phase'] = f'validation_{repeat + 1}_{path}'
        save_results()
        measured = run_benchmark(path, validation, 'validation_64', repeat + 1)
        result['validation_runs'].append(measured['summary'])
        if path not in result['validation_examples']:
            result['validation_examples'][path] = measured['examples']
        else:
            stable = examples_hash(measured['examples']) == examples_hash(result['validation_examples'][path])
            if not stable:
                result['phase'] = 'validation_repeat_changed'
                save_results()
                raise RuntimeError('Greedy changed between repeats; download diagnostics')
        trial[path] = measured['examples']
        save_results()
        print('VALIDATION', json.dumps(measured['summary']), flush=True)
    comparison = compare_examples(trial['reference'], trial['cached'])
    result['validation_comparisons'].append(comparison)
    save_results()
    if not comparison['all_tokens_equal']:
        result['phase'] = 'validation_parity_failed'
        save_results()
        raise RuntimeError('Reference and cache differ; run the download cell and return the ZIP')

medians = {path: statistics.median(s['seconds'] for s in result['validation_runs'] if s['path'] == path)
           for path in ['reference', 'cached']}
result['validation_median_seconds'] = medians
result['validation_speedup'] = medians['reference'] / medians['cached']
old_examples = previous_run['diagnostics']['50000']['validation_64']['examples']
result['reference_matches_saved_experiment06_code_count'] = sum(
    a['generated_code'] == b['generated_code'] for a, b in zip(result['validation_examples']['reference'], old_examples))
result['phase'] = 'validation_passed'
save_results()
print('VALIDATION_RESULT', json.dumps({'median_seconds': medians, 'speedup': result['validation_speedup'],
    'reference_equals_cached': True, 'reference_equals_saved_06_code_count': result['reference_matches_saved_experiment06_code_count']}))

In [ ]:
# 9. Complete test for BOTH paths; no timeout or early stop at 180 seconds.
assert result['phase'] == 'validation_passed', 'Finish validation parity first'
assert len(test_rows) == EXPECTED_TEST_ROWS
test_measured = {}
for path in ('reference', 'cached'):
    result['phase'] = f'full_test_{path}_running'
    save_results()
    print('START_FULL_TEST', path, len(test_rows), 'questions; reference can take several minutes', flush=True)
    test_measured[path] = run_benchmark(path, test_rows, 'test_1919', 1)
    result['test_runs'][path] = test_measured[path]['summary']
    save_results()
    print('FULL_TEST', json.dumps(result['test_runs'][path]), flush=True)

result['test_comparison'] = compare_examples(test_measured['reference']['examples'], test_measured['cached']['examples'])
result['test_speedup'] = result['test_runs']['reference']['seconds'] / result['test_runs']['cached']['seconds']
result['cached_test_within_180_seconds'] = result['test_runs']['cached']['seconds'] <= 180
result['model_fingerprint_after'] = model_fingerprint()
result['weights_unchanged'] = result['model_fingerprint_after'] == initial_model_fingerprint
assert result['weights_unchanged'], 'Inference changed model state'
test_output = {'experiment': result['experiment'], 'checkpoint_sha256': SOURCE_CHECKPOINT_SHA,
    'test_data': result['test_data'], 'comparison': result['test_comparison'],
    'reference': test_measured['reference']['examples'], 'cached': test_measured['cached']['examples']}
TEST_GENERATIONS_FILE.write_text(json.dumps(test_output, ensure_ascii=False, indent=2), encoding='utf-8')
result['test_generations_sha256'] = sha(TEST_GENERATIONS_FILE.read_bytes())
result['full_test_completed'] = True
result['run_finished'] = True
result['phase'] = 'finished' if result['test_comparison']['all_tokens_equal'] else 'finished_test_parity_failed'
save_results()
print('RESULT', json.dumps({'test_speedup': result['test_speedup'],
    'cached_test_seconds': result['test_runs']['cached']['seconds'],
    'cached_test_within_180_seconds': result['cached_test_within_180_seconds'],
    'identical_test_token_sequences': result['test_comparison']['identical_token_sequences'],
    'total_test_examples': len(test_rows), 'weights_unchanged': result['weights_unchanged']}))

In [ ]:
# 10. This cell also works after a stopped parity check; no weights in the ZIP.
save_results()
selected = [RESULTS_FILE]
if result.get('full_test_completed'):
    assert sha(TEST_GENERATIONS_FILE.read_bytes()) == result['test_generations_sha256']
    selected.append(TEST_GENERATIONS_FILE)
manifest = {'experiment': result['experiment'], 'phase': result['phase'],
    'run_finished': result['run_finished'], 'full_test_completed': result['full_test_completed'],
    'source_checkpoint_sha256': SOURCE_CHECKPOINT_SHA, 'notebook_with_outputs_included': False,
    'files': [{'filename': p.name, 'bytes': p.stat().st_size, 'sha256': sha(p.read_bytes())} for p in selected]}
with zipfile.ZipFile(ARCHIVE_FILE, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for p in selected:
        z.write(p, arcname=p.name)
    z.writestr('manifest.json', json.dumps(manifest, ensure_ascii=False, indent=2))
    z.writestr('README.txt', 'Experiment 07: fixed checkpoint50000, FP32, KV cache.\n'
        'Save the notebook WITH OUTPUTS separately. No weights included.\n'
        'Test generations are diagnostic, not an official bot submission.\n')
print('Download:', ARCHIVE_FILE.name, '| phase:', result['phase'])
from google.colab import files
files.download(str(ARCHIVE_FILE))